In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import urllib

In [2]:
server_name = r'.'
db_name = 'olist'
conn_str= ( "Driver={ODBC Driver 17 for SQL Server};" f"Server={server_name};" f"Database={db_name};" "Trusted_Connection=yes;" )

conn_params = urllib.parse.quote_plus(conn_str)
engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={conn_params}",
    fast_executemany=True
)

In [3]:
df1=pd.read_csv('olist_customers_dataset.csv')
df2 = pd.read_csv('olist_orders_dataset.csv')
df3 = pd.read_csv("olist_order_payments_dataset.csv")
df4 = pd.read_csv("olist_order_reviews_dataset.csv")
df5 = pd.read_csv("olist_sellers_dataset.csv")
df6 = pd.read_csv("product_category_name_translation.csv")
df7 = pd.read_csv("olist_products_dataset.csv")
df8 = pd.read_csv("olist_order_items_dataset.csv")

In [4]:
df1.to_sql('Customers', engine, if_exists='append', index=False, chunksize=500)
print('Done')

Done


In [5]:
df2['order_purchase_timestamp'] = pd.to_datetime(df2['order_purchase_timestamp'], errors='coerce')
df2['order_approved_at'] = pd.to_datetime(df2['order_approved_at'], errors='coerce')
df2['order_delivered_carrier_date'] = pd.to_datetime(df2['order_delivered_carrier_date'], errors='coerce')
df2['order_delivered_customer_date'] = pd.to_datetime(df2['order_delivered_customer_date'], errors='coerce')
df2['order_estimated_delivery_date'] = pd.to_datetime(df2['order_estimated_delivery_date'], errors='coerce')
df2.to_sql('Orders', engine, if_exists='append', index=False, chunksize=500)
print('Done')

Done


In [6]:
df3.to_sql('Order_Payments', engine, if_exists='append', index=False, chunksize=500)
print('Done')

Done


In [7]:
df4['review_id'].dropna(inplace=True)
df4.drop_duplicates(subset=['review_id'],inplace=True)
df4['review_creation_date'] = pd.to_datetime(df4['review_creation_date'],errors='coerce')
df4['review_answer_timestamp'] = pd.to_datetime(df4['review_answer_timestamp'],errors='coerce')
df4.to_sql('Order_Reviews', engine, if_exists='append', index=False, chunksize=500)
print('Done')

Done


In [8]:
df5.to_sql('Sellers', engine, if_exists='append', index=False, chunksize=500)
print('Done')

Done


In [9]:
translation_categories = set(df6['product_category_name'].unique())
print("Number of categories in translation:", len(translation_categories))

product_categories = set(df7['product_category_name'].dropna().unique())
print("Number of categories in products:", len(product_categories))

Number of categories in translation: 71
Number of categories in products: 73


In [10]:
missing_categories = product_categories - translation_categories
print("Categories lost in translation:", missing_categories)
print("Number of missing categories:", len(missing_categories))

null_count = df7['product_category_name'].isnull().sum()
print(f" null values: {null_count}")

Categories lost in translation: {'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'}
Number of missing categories: 2
 null values: 610


In [11]:
missing = pd.DataFrame([
    {'product_category_name': 'pc_gamer', 'product_category_name_english': 'pc_gamer'},
    {'product_category_name': 'portateis_cozinha_e_preparadores_de_alimentos', 'product_category_name_english': 'kitchen_portables'},
    {'product_category_name': 'unknown', 'product_category_name_english': 'unknown'}])

df_translation_updated = pd.concat([df6, missing], ignore_index=True)

In [12]:
df_translation_updated.to_sql('Category_Name_Translation', engine, if_exists='append', index=False, chunksize=500)
print('Done')

Done


In [13]:
df7['product_category_name'] = df7['product_category_name'].fillna('unknown')
df7.to_sql('Products', engine, if_exists='append', index=False, chunksize=500)
print('Done')

Done


In [14]:
df8.to_sql('Order_Items', engine, if_exists='append', index=False, chunksize=500)
print('Done')

Done
